In [2]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

from utils.models import CNNLSTM
from utils.preprocessing import normalize_label_name, load_and_preprocess_data
from utils.evaluation import (
    load_and_evaluate_rf_model,
    load_and_evaluate_cnnlstm_model,
)
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

## Preprocess data

In [ ]:
df_cicids2017 = load_and_preprocess_data("../data/CICIDS2017/wednesday_labeled.tsv")
df_ciciot2023 = load_and_preprocess_data("../data/CICIoT2023/ciciot2023_labeled_conn.tsv")
dataframes = [df_cicids2017, df_ciciot2023]

In [ ]:
for df in dataframes:
    for col in [
        "duration",
        "orig_bytes",
        "resp_bytes",
        "missed_bytes",
        "orig_pkts",
        "orig_ip_bytes",
        "resp_pkts",
        "resp_ip_bytes",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

In [ ]:
label_column = "label"

df_cicids2017 = df_cicids2017[df_cicids2017["label"].isin(["DOS_HTTP_FLOOD"] + ["BENIGN"])]
df_ciciot2023 = df_ciciot2023[df_ciciot2023["label"].isin(["DOS_HTTP_FLOOD"] + ["BENIGN"])]

X_cicids = df_cicids2017.drop(columns=[label_column])
y_cicids = df_cicids2017[label_column]

X_ciciot = df_ciciot2023.drop(columns=[label_column])
y_ciciot = df_ciciot2023[label_column]

print("CICIDS test shape:", X_cicids.shape)
print("CICIoT test shape:", X_ciciot.shape)

CICIDS test shape: (481829, 22)
CICIoT test shape: (1850844, 22)


## Load Models and evaluate

In [ ]:
# Test CICIDS2017-trained RF model on CICIoT2023
load_and_evaluate_rf_model(
    joblib_path="models/random_forest_cicids2017.joblib",
    X=X_ciciot,
    y_true=y_ciciot,
    model_name="CICIDS2017-trained RF on CICIoT2023"
)

Evaluation for CICIDS2017-trained RF on CICIoT2023:


AttributeError: 'RandomForestClassifier' object has no attribute 'feature_names_'

In [ ]:
# Test CICIoT2023-trained RF model on CICIDS2017
load_and_evaluate_rf_model(
    joblib_path="models/random_forest_ciciot2023.joblib",
    X=X_cicids,
    y_true=y_cicids,
    model_name="CICIoT2023-trained RF on CICIDS2017"
)

In [ ]:
# Test CICIoT2023-trained CNN-LSTM model on CICIDS2017
load_and_evaluate_cnnlstm_model(
    joblib_path="models/cnnlstm_ciciot2023.joblib",
    X=X_cicids,
    y_true=y_cicids,
    model_name="CICIoT2023-trained CNN-LSTM on CICIDS2017"
)

In [ ]:
# Test CICIDS2017-trained CNN-LSTM model on CICIoT2023
load_and_evaluate_cnnlstm_model(
    joblib_path="models/cnnlstm_cicids2017.joblib",
    X=X_ciciot,
    y_true=y_ciciot,
    model_name="CICIDS2017-trained CNN-LSTM on CICIoT2023"
)